# ClaimLens: fine-tune YOLO11-seg on CarDD

Trains the car-damage detector used by `services/vision`.

1. Mount Google Drive (CarDD lives there as a shortcut).
2. Convert CarDD from COCO to YOLO-seg format with `training/convert_cardd.py`, onto Colab's local disk.
3. Fine-tune `yolo11s-seg` (COCO-pretrained) on the 6 damage classes.
4. Evaluate on the held-out **test** split: box and mask mAP@0.5 and mAP@0.5:0.95, overall and per class.
5. Export `best.pt` and `best.onnx`.

**Why segmentation, not just boxes?** CarDD ships polygon masks. A seg model predicts a mask per damage, so the
vision service can compute damage area % from pixels instead of box area (a diagonal scratch fills a tiny part of its box).

**Runtime:** Runtime → Change runtime type → **T4 GPU**. Runs and weights are saved to Drive, so if Colab disconnects,
re-run all cells and training resumes from the last epoch.

**Data licence:** CarDD is for research and education only. Don't copy it out of Drive or redistribute it.

## 1. Config

In [ ]:
# Edit these to match your Drive.
# Where CarDD is. If it's a .zip in your Drive, leave both empty: step 2b finds
# any .zip with "cardd" in its name and unzips it. Otherwise set one of these:
CARDD_ZIP = ""  # the zip's full path
CARDD_ROOT = ""  # an already-unzipped folder (the one holding annotations/)
RUNS_DIR = "/content/drive/MyDrive/claimlens/runs"  # on Drive so a disconnect doesn't lose weights
REPO_URL = "https://github.com/AndrewRaj0510/ClaimLens.git"
DATASET_DIR = "/content/cardd_yolo"  # local disk: reading thousands of images from Drive each epoch is slow

MODEL = "yolo11s-seg.pt"  # 's' fits a T4 comfortably; try 'm' later if there's time
RUN_NAME = "cardd_yolo11s_seg"
EPOCHS = 100
IMGSZ = 640
BATCH = 16
PATIENCE = 20  # stop early if val mAP hasn't improved for this many epochs
SEED = 0

In [ ]:
!nvidia-smi

## 2. Mount Drive

If CarDD was shared with you, it only appears under `MyDrive` after you right-click it → *Organize* → *Add shortcut*.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2b. Unzip CarDD

Copies the zip from Drive to Colab's own disk and unzips it there (reading from the
copy is much faster than from Drive). Takes a few minutes. After a disconnect it runs
again, because Colab's disk is wiped; your training progress on Drive is not.

In [ ]:
import glob
import shutil
import zipfile
from pathlib import Path

DRIVE = "/content/drive/MyDrive"
EXTRACT_DIR = Path("/content/CarDD")


def find_cardd_root(base):
    """The folder that holds annotations/instances_train2017.json, wherever the zip put it."""
    hits = [
        p
        for p in Path(base).rglob("instances_train2017.json")
        if p.parent.name == "annotations" and "__MACOSX" not in p.parts
    ]
    return str(min(hits).parent.parent) if hits else None


if CARDD_ROOT and Path(CARDD_ROOT, "annotations").is_dir():
    print("Using the unzipped CarDD folder:", CARDD_ROOT)
else:
    CARDD_ROOT = find_cardd_root(EXTRACT_DIR) if EXTRACT_DIR.exists() else None
    if CARDD_ROOT:
        print("Already unzipped in this session.")
    else:
        zip_path = CARDD_ZIP
        if not zip_path:
            zips = sorted(
                {p for d in ("", "*/", "*/*/") for p in glob.glob(f"{DRIVE}/{d}*.zip")}
            )
            print("Zip files in your Drive (3 folders deep):", *zips, sep="\n  ")
            matches = [z for z in zips if "cardd" in Path(z).name.lower()]
            assert len(matches) == 1, (
                "Couldn't pick the CarDD zip automatically. Copy its path from the list "
                "above into CARDD_ZIP in the config cell, run that cell, then this one."
            )
            zip_path = matches[0]
        local_zip = Path("/content/CarDD.zip")
        print(f"Copying {zip_path} to Colab's disk...")
        shutil.copyfile(zip_path, local_zip)
        print("Unzipping...")
        with zipfile.ZipFile(local_zip) as z:
            z.extractall(EXTRACT_DIR)
        local_zip.unlink()  # free the disk space
        CARDD_ROOT = find_cardd_root(EXTRACT_DIR)

assert CARDD_ROOT, (
    "Unzipped, but there's no annotations/instances_train2017.json inside. "
    "Run  !ls -R /content/CarDD | head -60  in a new cell and send the output."
)
print("CARDD_ROOT =", CARDD_ROOT)

## 3. Install Ultralytics and fetch the repo code

In [ ]:
%pip install -q "ultralytics>=8.3,<9"
import ultralytics

ultralytics.checks()

In [ ]:
import os
import sys

# Pulls convert_cardd.py from GitHub, so push your latest training/ changes first.
if not os.path.exists("/content/ClaimLens"):
    !git clone --depth 1 {REPO_URL} /content/ClaimLens
sys.path.insert(0, "/content/ClaimLens/training")

## 4. Convert CarDD → YOLO-seg

Expected counts for the official release: train 2816, val 810, test 374 images.

In [ ]:
from convert_cardd import convert

names, stats = convert(CARDD_ROOT, DATASET_DIR)
print("classes:", names)
for split, s in stats.items():
    print(f"{split:>5}: {dict(s)}")

assert len(names) == 6, f"expected 6 CarDD classes, got {names}"
assert not any(s["missing_images"] for s in stats.values()), (
    "some images listed in the JSON were not found"
)

## 5. Train

Safe to re-run: it trains from scratch, resumes an interrupted run, or skips if the run already finished.
After training starts, check `train_batch0.jpg` in the run folder to confirm the masks line up with the damage.

In [ ]:
import json
from pathlib import Path

from ultralytics import YOLO

run_dir = Path(RUNS_DIR) / RUN_NAME
last = run_dir / "weights" / "last.pt"
# Written after train() returns. Without it, "resume" on a finished run makes
# Ultralytics silently start a new run on its demo dataset.
done = run_dir / "training_complete.json"

if done.exists():
    print(f"Already trained: {done.read_text()}")
elif last.exists():
    # Colab disconnected mid-run: continue from the last saved epoch with the same settings.
    model = YOLO(str(last))
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    model.train(
        data=f"{DATASET_DIR}/data.yaml",
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=PATIENCE,
        seed=SEED,
        deterministic=True,
        project=RUNS_DIR,
        name=RUN_NAME,
        exist_ok=True,
    )

if not done.exists():
    done.write_text(json.dumps({"model": MODEL, "epochs_max": EPOCHS, "imgsz": IMGSZ}))

## 6. Evaluate on the test split

Model selection (`best.pt`) used the **val** split, so we report on **test**, which the model has never influenced.

- **mAP@0.5**: average precision with a prediction counted correct at IoU ≥ 0.5. Lenient on localisation.
- **mAP@0.5:0.95**: mean over IoU thresholds 0.5 to 0.95. Rewards tight boxes/masks; the stricter headline number.
- **box** vs **mask**: the same metrics computed on bounding boxes vs pixel masks.

In [ ]:
best = run_dir / "weights" / "best.pt"
model = YOLO(str(best))
metrics = model.val(
    data=f"{DATASET_DIR}/data.yaml",
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    project=RUNS_DIR,
    name=f"{RUN_NAME}_test",
    exist_ok=True,
)

per_class = {}
for i, c in enumerate(metrics.box.ap_class_index):
    _, _, box50, box = metrics.box.class_result(i)
    _, _, seg50, seg = metrics.seg.class_result(i)
    per_class[model.names[int(c)]] = {
        "box_map50": float(box50),
        "box_map50_95": float(box),
        "mask_map50": float(seg50),
        "mask_map50_95": float(seg),
    }

results = {
    "model": MODEL,
    "run": RUN_NAME,
    "ultralytics": ultralytics.__version__,
    "imgsz": IMGSZ,
    "epochs_max": EPOCHS,
    "seed": SEED,
    "test_images": stats["test"]["images"],
    "box_map50": float(metrics.box.map50),
    "box_map50_95": float(metrics.box.map),
    "mask_map50": float(metrics.seg.map50),
    "mask_map50_95": float(metrics.seg.map),
    "per_class": per_class,
}
(run_dir / "test_metrics.json").write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))

Markdown table to paste into `docs/metrics.md`:

In [ ]:
rows = [("**all**", results)] + list(per_class.items())
print(
    f"Test split ({results['test_images']} images), {MODEL}, imgsz {IMGSZ}, ultralytics {results['ultralytics']}\n"
)
print("| class | box mAP@0.5 | box mAP@0.5:0.95 | mask mAP@0.5 | mask mAP@0.5:0.95 |")
print("|---|---|---|---|---|")
for name, r in rows:
    print(
        f"| {name} | {r['box_map50']:.3f} | {r['box_map50_95']:.3f} "
        f"| {r['mask_map50']:.3f} | {r['mask_map50_95']:.3f} |"
    )

## 7. Eyeball a few test predictions

In [ ]:
import random

import matplotlib.pyplot as plt

test_images = sorted(Path(DATASET_DIR, "images", "test").glob("*.jpg"))
random.Random(SEED).shuffle(test_images)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, img in zip(axes.flat, test_images[:6]):
    pred = model.predict(str(img), imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    ax.imshow(pred.plot()[:, :, ::-1])  # plot() returns BGR
    ax.set_title(img.name)
    ax.axis("off")
plt.tight_layout()

## 8. Export ONNX

ONNX lets the vision service run on `onnxruntime` (CPU or GPU) without PyTorch, giving a much smaller Docker image.
`best.onnx` is written next to `best.pt` in the run's `weights/` folder on Drive.

In [ ]:
onnx_path = model.export(format="onnx", imgsz=IMGSZ, simplify=True)
print(onnx_path)

## Next

- Download `best.pt` and `best.onnx` from `RUNS_DIR/RUN_NAME/weights/`. Weights are git-ignored, so keep them out of the repo.
- Paste the table above into `docs/metrics.md` as the baseline.